# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and processing the FAIR² dataset—"Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells"—using the `mlcroissant` library.

### Dataset Source

The dataset is defined by a [FAIR croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading

We first load the dataset metadata using the Croissant schema URL, inspect basic metadata, and check dataset contents.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata. Do NOT subscript, only access as an object
meta = dataset.metadata
print(f"Dataset Title: {meta.name}\n\nDescription: {meta.description}\n")

# List available record sets
print("Available Record Sets (@id):")
record_sets = meta.record_sets
for rset in record_sets:
    print(f"- {rset.id}: {rset.name if hasattr(rset, 'name') else ''}")

if len(record_sets) == 0:
    print("No record sets explicitly listed in metadata, attempting to infer from dataset API.")

## 2. Data Overview

Review all available record sets and their fields (columns) referenced by their `@id`. This is important for mapping further data extractions and transformations.


In [ ]:
# Since many croissant datasets only have a default/main table, let's list all top-level record sets and their field IDs

if len(record_sets) == 0:
    # Try direct call through API to list record_set candidates
    # The dataset API usually supports .record_set_ids
    record_set_ids = dataset.record_set_ids
    print("RecordSet @id(s):", record_set_ids)
else:
    record_set_ids = [rset.id for rset in record_sets]
    print("RecordSet @id(s):", record_set_ids)

# Explore field ids for each record set
for rs_id in record_set_ids:
    print(f"\nRecordSet: {rs_id}\nFields (by @id):")
    # Get the fields for the record set
    fields = dataset.get_record_set_fields(rs_id)
    for field in fields:
        # field.id, field.name, field.data_type, etc
        print(f"  - {field.id}\t| {field.name if hasattr(field, 'name') else ''}\t| {getattr(field, 'data_type', '')}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For demonstration, extract the first record set (main table) found in the list.


In [ ]:
dfs = dict()

# For most croissant datasets, first record set is primary
main_rs_id = record_set_ids[0]
print(f"Extracting main data table from RecordSet @id: {main_rs_id}")

records = list(dataset.records(record_set=main_rs_id))
df = pd.DataFrame(records)
dfs[main_rs_id] = df
print(f"Loaded {len(df)} records.")

print(f"Columns: {list(df.columns)}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's filter records, normalize a numeric value, examine summary statistics, and group data by relevant categorical fields.

**You may wish to adapt field IDs below to match your research purpose as explored in Section 2.**

In [ ]:
# Choose a likely numeric field id by inspection (update as per the actual main table columns)
# Common examples: coverage, peptide_count, molecular_weight

possible_numeric_fields = [
    col for col in df.columns if any(x in col.lower() for x in ['coverage', 'weight', 'count', 'abundance', 'number', 'peptide'])
]
print(f"Detected numeric-like fields: {possible_numeric_fields}")

# Select first found (or specify):
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]

# Show basic stats
print(df[numeric_field].describe())

# Attempt to coerce to numeric, if not already
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Choose a threshold (e.g., above median)
threshold = df[numeric_field].median()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold}")

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field
possible_group_fields = [
    col for col in df.columns if col != numeric_field and df[col].nunique() < max(8, len(df)//25)
]
print(f"Possible group fields: {possible_group_fields}")
if possible_group_fields:
    group_field = possible_group_fields[0]
    grouped = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Group mean of {numeric_field} by {group_field}:")
    print(grouped.head())

## 5. Visualization

Plotting distributions and relationships using the extracted and processed fields. You may use any of the detected field `@id` column names.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of a numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If grouping by a field, show boxplot
if possible_group_fields:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion

- We loaded and explored protein mass spectrometry data using the `mlcroissant` library, referencing all subsets and columns by their Croissant `@id`.
- Data was filtered, normalized, summarized, and visualized.
- For further biological or analytical insights, continue to leverage the Croissant field and record set `@id`s and explore the dataset's documentation.
